In [1]:
# =========================================================
# LEVEL 3 - SUPPORT VECTOR MACHINE (SVM)
# IRIS DATASET
# =========================================================


# =========================================================
# 1. IMPORT LIBRARIES
# =========================================================

import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV
)

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# =========================================================
# 2. LOAD DATASET
# =========================================================

df = pd.read_csv("dataset/1) iris.csv")


# =========================================================
# 3. EXPLORE DATASET
# =========================================================

print("First 5 Rows:")
display(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nDataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nClass Distribution:")
print(df["species"].value_counts())


# =========================================================
# 4. SEPARATE FEATURES AND TARGET
# =========================================================

X = df.drop(columns=["species"])
y = df["species"]

print("\nFeatures:")
display(X.head())

print("\nTarget:")
display(y.head())


# =========================================================
# 5. TRAIN / TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining Shape:")
print(X_train.shape)

print("\nTesting Shape:")
print(X_test.shape)


# =========================================================
# 6. SCALE DATA FOR BASELINE MODEL
# =========================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)


# =========================================================
# 7. CREATE BASELINE SVM MODEL
# =========================================================

baseline_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale"
)


# =========================================================
# 8. TRAIN BASELINE MODEL
# =========================================================

baseline_model.fit(
    X_train_scaled,
    y_train
)


# =========================================================
# 9. BASELINE PREDICTIONS
# =========================================================

baseline_y_pred = baseline_model.predict(
    X_test_scaled
)


# =========================================================
# 10. ACTUAL VS PREDICTED
# =========================================================

comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": baseline_y_pred
})

print("\nActual vs Predicted:")
display(comparison.head(10))


# =========================================================
# 11. BASELINE MODEL EVALUATION
# =========================================================

baseline_accuracy = accuracy_score(
    y_test,
    baseline_y_pred
)

baseline_precision = precision_score(
    y_test,
    baseline_y_pred,
    average="weighted"
)

baseline_recall = recall_score(
    y_test,
    baseline_y_pred,
    average="weighted"
)

baseline_f1 = f1_score(
    y_test,
    baseline_y_pred,
    average="weighted"
)

print("\n========== BASELINE SVM ==========")

print("Accuracy :", baseline_accuracy)
print("Precision:", baseline_precision)
print("Recall   :", baseline_recall)
print("F1 Score :", baseline_f1)


# =========================================================
# 12. BASELINE CONFUSION MATRIX
# =========================================================

baseline_cm = confusion_matrix(
    y_test,
    baseline_y_pred
)

print("\nBaseline Confusion Matrix:")
print(baseline_cm)


# =========================================================
# 13. BASELINE CLASSIFICATION REPORT
# =========================================================

print("\nBaseline Classification Report:")

print(
    classification_report(
        y_test,
        baseline_y_pred
    )
)


# =========================================================
# 14. NUMBER OF SUPPORT VECTORS
# =========================================================

print("\nNumber of Support Vectors Per Class:")

print(
    baseline_model.n_support_
)

print("\nTotal Number of Support Vectors:")

print(
    baseline_model.n_support_.sum()
)


# =========================================================
# 15. TEST DIFFERENT KERNELS
# =========================================================

kernels = [
    "linear",
    "rbf",
    "poly",
    "sigmoid"
]

kernel_results = []

print("\n========== KERNEL COMPARISON ==========")

for kernel in kernels:

    kernel_model = SVC(
        kernel=kernel
    )

    kernel_model.fit(
        X_train_scaled,
        y_train
    )

    kernel_prediction = kernel_model.predict(
        X_test_scaled
    )

    kernel_accuracy = accuracy_score(
        y_test,
        kernel_prediction
    )

    kernel_f1 = f1_score(
        y_test,
        kernel_prediction,
        average="weighted"
    )

    kernel_results.append({
        "Kernel": kernel,
        "Accuracy": kernel_accuracy,
        "F1 Score": kernel_f1
    })


kernel_results_df = pd.DataFrame(
    kernel_results
)

display(kernel_results_df)


# =========================================================
# 16. BASELINE CROSS VALIDATION
# =========================================================

# Pipeline is used so StandardScaler is fitted
# separately inside every cross-validation fold.

baseline_pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "svm",
        SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale"
        )
    )
])


cv_scores = cross_val_score(
    baseline_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="f1_weighted"
)

print("\n========== 5-FOLD CROSS VALIDATION ==========")

print("Cross Validation F1 Scores:")
print(cv_scores)

print("\nMean CV F1 Score:")
print(cv_scores.mean())

print("\nCV Standard Deviation:")
print(cv_scores.std())


# =========================================================
# 17. CREATE PIPELINE FOR HYPERPARAMETER TUNING
# =========================================================

pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "svm",
        SVC()
    )
])


# =========================================================
# 18. DEFINE HYPERPARAMETERS
# =========================================================

# Linear kernel does not need gamma.
# RBF kernel uses both C and gamma.

param_grid = [

    {
        "svm__kernel": ["linear"],

        "svm__C": [
            0.1,
            1,
            10,
            100
        ]
    },

    {
        "svm__kernel": ["rbf"],

        "svm__C": [
            0.1,
            1,
            10,
            100
        ],

        "svm__gamma": [
            "scale",
            0.01,
            0.1,
            1
        ]
    }

]


# =========================================================
# 19. CREATE GRID SEARCH
# =========================================================

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1_weighted",
    n_jobs=-1
)


# =========================================================
# 20. RUN HYPERPARAMETER TUNING
# =========================================================

grid_search.fit(
    X_train,
    y_train
)


# =========================================================
# 21. DISPLAY BEST PARAMETERS
# =========================================================

print("\n========== GRID SEARCH RESULTS ==========")

print("\nBest Parameters:")

print(
    grid_search.best_params_
)

print("\nBest Cross Validation F1 Score:")

print(
    grid_search.best_score_
)


# =========================================================
# 22. GET BEST MODEL
# =========================================================

best_model = grid_search.best_estimator_


# =========================================================
# 23. MAKE PREDICTIONS WITH BEST MODEL
# =========================================================

# Important:
# We use X_test directly, not X_test_scaled.
#
# The Pipeline automatically applies:
#
# StandardScaler
#       ↓
# Best SVM

best_y_pred = best_model.predict(
    X_test
)


# =========================================================
# 24. EVALUATE TUNED MODEL
# =========================================================

best_accuracy = accuracy_score(
    y_test,
    best_y_pred
)

best_precision = precision_score(
    y_test,
    best_y_pred,
    average="weighted"
)

best_recall = recall_score(
    y_test,
    best_y_pred,
    average="weighted"
)

best_f1 = f1_score(
    y_test,
    best_y_pred,
    average="weighted"
)

print("\n========== TUNED SVM ==========")

print("Accuracy :", best_accuracy)

print("Precision:", best_precision)

print("Recall   :", best_recall)

print("F1 Score :", best_f1)


# =========================================================
# 25. TUNED CONFUSION MATRIX
# =========================================================

best_cm = confusion_matrix(
    y_test,
    best_y_pred
)

print("\nTuned Confusion Matrix:")

print(
    best_cm
)


# =========================================================
# 26. TUNED CLASSIFICATION REPORT
# =========================================================

print("\nTuned Classification Report:")

print(
    classification_report(
        y_test,
        best_y_pred
    )
)


# =========================================================
# 27. BASELINE VS TUNED MODEL
# =========================================================

model_comparison = pd.DataFrame({

    "Model": [
        "Baseline SVM",
        "Tuned SVM"
    ],

    "Accuracy": [
        baseline_accuracy,
        best_accuracy
    ],

    "Precision": [
        baseline_precision,
        best_precision
    ],

    "Recall": [
        baseline_recall,
        best_recall
    ],

    "F1 Score": [
        baseline_f1,
        best_f1
    ]

})

print("\n========== MODEL COMPARISON ==========")

display(
    model_comparison
)


# =========================================================
# 28. FINAL RESULTS
# =========================================================

print("\n========== FINAL RESULTS ==========")

print("\nBaseline SVM:")

print("Accuracy :", baseline_accuracy)
print("Precision:", baseline_precision)
print("Recall   :", baseline_recall)
print("F1 Score :", baseline_f1)


print("\nBaseline 5-Fold CV:")

print(
    cv_scores
)

print(
    "Mean CV F1:",
    cv_scores.mean()
)


print("\nBest Hyperparameters:")

print(
    grid_search.best_params_
)


print("\nTuned SVM:")

print("Accuracy :", best_accuracy)
print("Precision:", best_precision)
print("Recall   :", best_recall)
print("F1 Score :", best_f1)


print(
    "\nSVM Task Completed Successfully."
)

First 5 Rows:


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa



Dataset Shape:
(150, 5)

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
 4   species       150 non-null    str    
dtypes: float64(4), str(1)
memory usage: 6.0 KB

Missing Values:
sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
species         0
dtype: int64

Class Distribution:
species
setosa        50
versicolor    50
virginica     50
Name: count, dtype: int64

Features:


,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2



Target:


...


========== BASELINE SVM ==========
Accuracy : 0.9666666666666667
Precision: 0.9696969696969696
Recall   : 0.9666666666666667
F1 Score : 0.9665831244778613

Baseline Confusion Matrix:
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]

Baseline Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30


Number of Support Vectors Per Class:
[10 18 19]

Total Number of Support Vectors:
47

========== KERNEL COMPARISON ==========


,Kernel,Accuracy,F1 Score
0,linear,1.000000,1.000000
1,rbf,0.966667,0.966583
2,poly,0.900000,0.897698
3,sigmoid,0.900000,0.897698



========== 5-FOLD CROSS VALIDATION ==========
Cross Validation F1 Scores:
[0.91534392 1.         0.95816993 0.95816993 1.        ]

Mean CV F1 Score:
0.9663367569249923

CV Standard Deviation:
0.03162306668752773

========== GRID SEARCH RESULTS ==========

Best Parameters:
{'svm__C': 1, 'svm__kernel': 'linear'}

Best Cross Validation F1 Score:
0.9749019607843138

========== TUNED SVM ==========
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

Tuned Confusion Matrix:
[[10  0  0]
 [ 0 10  0]
 [ 0  0 10]]

Tuned Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00        10
   virginica       1.00      1.00      1.00        10

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30


========== MODEL COMPARISON ==========


,Model,Accuracy,Precision,Recall,F1 Score
0,Baseline SVM,0.966667,0.969697,0.966667,0.966583
1,Tuned SVM,1.000000,1.000000,1.000000,1.000000



========== FINAL RESULTS ==========

Baseline SVM:
Accuracy : 0.9666666666666667
Precision: 0.9696969696969696
Recall   : 0.9666666666666667
F1 Score : 0.9665831244778613

Baseline 5-Fold CV:
[0.91534392 1.         0.95816993 0.95816993 1.        ]
Mean CV F1: 0.9663367569249923

Best Hyperparameters:
{'svm__C': 1, 'svm__kernel': 'linear'}

Tuned SVM:
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

SVM Task Completed Successfully.
